## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# Parte 1

### CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,46178862814,3381089,2026-06-30,2026-06-30,2,E,63636.500000000,HVA3,3381089,"{""pessoas"":[{""nome"":""AMANDA LAIS GRECO GARCIA""...",...,34,1778785248777,"{""valor_aluguel"":1900,""matchmaking_on"":false,""...",2026-06-30 18:00:46+00:00,1782842445188104912,46178862814,C,2026-06-30 18:02:28.755000+00:00,2026-06-30 14:51:22+00:00,APROVAR
1,34117028855,4130577,2026-06-09,2026-06-09,1,NI,1986.500000000,BLEND3_3,4130577,"{""pessoas"":[{""nome"":""RAFAEL RODRIGO MARQUES"",""...",...,34,1778785248777,"{""valor_aluguel"":""1400"",""imobiliaria_id"":18803...",2026-06-09 18:00:43+00:00,1781028038218248120,34117028855,C,2026-06-09 18:09:01.409000+00:00,2026-06-09 11:19:52+00:00,APROVAR
2,22189750172,4138492,2026-06-12,2026-06-12,1,A,13357.500000000,BLEND3_3,4138492,"{""pessoas"":[{""nome"":""RAIMUNDO NONATO BORGES DA...",...,32,1778785248777,"{""valor_aluguel"":""1300"",""imobiliaria_id"":18121...",2026-06-12 18:00:36+00:00,1781287231125549233,22189750172,B,2026-06-12 18:05:19.778000+00:00,2026-06-12 10:33:08+00:00,APROVAR
3,3659237450,4161529,2026-06-15,2026-06-15,1,E,1918.000000000,BLEND3_3,4161529,"{""pessoas"":[{""nome"":""ANA CELIA DA CONCEICAO"",""...",...,34,1778785248777,"{""valor_aluguel"":""1300"",""imobiliaria_id"":13975...",2026-06-15 18:00:36+00:00,1781546433132420163,03659237450,C,2026-06-15 18:01:20.892000+00:00,2026-06-15 13:21:38+00:00,APROVAR
4,49177067851,4161897,2026-06-16,2026-06-16,1,C,1507.000000000,BLEND3_3,4161897,"{""pessoas"":[{""nome"":""AMANDA GLECIA DE AQUINO"",...",...,33,1778785248777,"{""valor_aluguel"":""1600"",""imobiliaria_id"":33067...",2026-06-16 18:00:37+00:00,1781632836177182585,49177067851,E,2026-06-16 18:03:54.415000+00:00,2026-06-16 14:36:15+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114036,11162125870,4333583,2026-07-04,2026-07-04,1,NI,61787.000000000,BLEND3_3,4333583,"{""pessoas"":[{""nome"":""FLAVIO LEPKA CORREIA DA S...",...,31,1778785248777,"{""valor_aluguel"":4300,""matchmaking_on"":false,""...",2026-07-05 08:00:25+00:00,1783238424029094282,11162125870,A,2026-07-05 08:08:37.959000+00:00,2026-07-04 20:04:13+00:00,APROVAR
114037,66190525504,4333674,2026-07-05,2026-07-05,1,NI,12193.000000000,BLEND3_3,4333674,"{""pessoas"":[{""nome"":""EDMILSON DOS SANTOS LIMA""...",...,33,1778785248777,"{""imobiliaria_id"":""22006"",""imovel"":{""tipo"":""RE...",2026-07-05 18:00:33+00:00,1783274432541840374,66190525504,E,2026-07-05 18:02:54.383000+00:00,2026-07-05 12:13:15+00:00,APROVAR
114038,8750143999,4333678,2026-07-05,2026-07-05,1,C,1027.500000000,BLEND3_3,4333678,"{""pessoas"":[{""nome"":""JULIANA GUEDES ANTUNES"",""...",...,33,1778785248777,"{""valor_aluguel"":1450,""matchmaking_on"":false,""...",2026-07-05 18:00:33+00:00,1783274432553611453,08750143999,E,2026-07-05 18:02:54.386000+00:00,2026-07-05 12:21:56+00:00,APROVAR
114039,60127399313,4333774,2026-07-05,2026-07-05,1,D,1575.500000000,BLEND3_3,4333774,"{""pessoas"":[{""nome"":""GESSICA RODRIGUES"",""docum...",...,34,1778785248777,"{""valor_aluguel"":900,""matchmaking_on"":false,""p...",2026-07-06 08:00:33+00:00,1783324831745504223,60127399313,C,2026-07-06 08:03:54.989000+00:00,2026-07-05 19:37:22+00:00,APROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                97591
BLEND_REGRESSAO_2026     9053
HVA3                     4526
BVS_CUSTOM               1803
HVA4                      665
BLEND_4                   373
BVS_CUSTOM_V2              14
                           13
HFT1                        3
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    109611
2      4090
3       305
4        33
6         1
5         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961154
2    0.035864
3    0.002674
4    0.000289
6    0.000009
5    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

contrato_df = (
    pd.json_normalize(payload_parsed, sep="_")
    .add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.
)

pessoa_df = pd.json_normalize(
    payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0]),
    sep="_",
).add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

In [13]:
credpago_expandido = (
    credpago_df.loc[valid]
    .join(contrato_df, how="left")   # contrato_df index = parsed.dropna().index
    .join(pessoa_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)

In [14]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])

### CredPago - Imóvel e Histórico de Valor

In [15]:
query = """
WITH property_type AS (
  SELECT
    id AS contract_id,
    CASE WHEN tp_imovel = 2 THEN 0 ELSE 1 END AS property_type,
    CAST(id_cidade_ibge AS INT64) AS id_cidade_ibge,    --nova


    TRIM(
    REGEXP_REPLACE(
        REGEXP_REPLACE(
        UPPER(
            REGEXP_REPLACE(
            NORMALIZE(COALESCE(cidade, ''), NFD),
            r'\pM', ''                      -- tira acento (marcas)
            )
        ),
        r'[^A-Z0-9 ]', ' '                 -- tira lixo (agora já está tudo em maiúsculo)
        ),
        r'\s+', ' '                          -- colapsa espaços
    )
    ) AS contract_city_nm, --nova

    CAST(id_imobiliaria AS INT64) AS agency_id,  --nova
    DATE(criado_em) AS requested_at
  FROM `loft-dl-fintech.bronze_credpago.imovel`
  WHERE DATE(criado_em) >= DATE('2026-01-01')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY criado_em DESC) = 1
),

rental_value AS (
  SELECT
    id_imovel AS contract_id,
    CAST(vl_aluguel AS FLOAT64) AS rental_value,
    DATE(data) AS requested_at
  FROM `loft-dl-fintech.bronze_credpago.historico_valor`
  WHERE DATE(data) >= DATE('2026-01-01')
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id_imovel ORDER BY data DESC) = 1
)

SELECT
  COALESCE(p.contract_id, r.contract_id) AS contract_id,
  GREATEST(p.requested_at, r.requested_at) AS requested_at,
  p.property_type,
  p.id_cidade_ibge,
  p.contract_city_nm,
  p.agency_id,
  r.rental_value
FROM property_type p
FULL OUTER JOIN rental_value r
  ON p.contract_id = r.contract_id;
"""

credpago_imovel_df = pd.read_gbq(query, project_id=project_id)

In [16]:
credpago_imovel_df.head()

,contract_id,requested_at,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value
0,3568906,2026-01-01,1,4300604,ALVORADA,2687,1100.0
1,3569023,2026-01-02,1,3205309,VITORIA,24904,2200.0
2,3569164,2026-01-02,1,4202107,BARRA VELHA,14077,1800.0
3,3569393,2026-01-02,0,1100205,PORTO VELHO,28288,1500.0
4,3569690,2026-01-02,1,4318309,SAO GABRIEL,32242,600.0


### Merge 

In [17]:
cp_final_df = pd.merge(credpago_clean, credpago_imovel_df.drop(columns=['requested_at']), on='contract_id', how='left')
cp_final_df.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,request,...,pessoa_ratings_BVS_CUSTOM_V2,pessoa_ratings_HFT1,qtde_pefin,valor_pefin_total,modalidades_pefin,property_type,id_cidade_ibge,contract_city_nm,agency_id,rental_value
0,46178862814,3381089,2026-06-30,2026-06-30,2,E,HVA3,6023568,34,"{""valor_aluguel"":1900,""matchmaking_on"":false,""...",...,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,NaN
1,34117028855,4130577,2026-06-09,2026-06-09,1,NI,BLEND3_3,5917259,34,"{""valor_aluguel"":""1400"",""imobiliaria_id"":18803...",...,NaN,NaN,1.0,1327.0,CRED CARTAO,1,3554102,TAUBATE,18803,1400.0
2,22189750172,4138492,2026-06-12,2026-06-12,1,A,BLEND3_3,5939236,32,"{""valor_aluguel"":""1300"",""imobiliaria_id"":18121...",...,NaN,NaN,NaN,NaN,NaN,1,5300108,BRASILIA,18121,1300.0
3,3659237450,4161529,2026-06-15,2026-06-15,1,E,BLEND3_3,5947530,34,"{""valor_aluguel"":""1300"",""imobiliaria_id"":13975...",...,NaN,NaN,NaN,NaN,NaN,1,3526407,LARANJAL PAULISTA,13975,1300.0
4,49177067851,4161897,2026-06-16,2026-06-16,1,C,BLEND3_3,5955606,33,"{""valor_aluguel"":""1600"",""imobiliaria_id"":33067...",...,NaN,NaN,NaN,NaN,NaN,1,3526902,LIMEIRA,33067,1600.0


In [18]:
cols_ = ["property_type", "rental_value", "id_cidade_ibge", "agency_id"]

pct_nulls = (
    cp_final_df[cols_].isna().mean().mul(100).sort_values(ascending=False)
)
pct_nulls

rental_value      5.282311
property_type     1.260073
id_cidade_ibge    1.260073
agency_id         1.260073
dtype: float64

In [19]:
pct_nulls_por_requested_at = (
    cp_final_df.groupby("requested_at")[cols_]
    .apply(lambda x: x.isna().mean().mul(100))
    .sort_index()
)

pct_nulls_por_requested_at

,property_type,rental_value,id_cidade_ibge,agency_id
requested_at,,,,
2026-06-08,0.000000,0.000000,0.000000,0.000000
2026-06-09,0.012509,0.000000,0.012509,0.012509
2026-06-10,0.017937,0.017937,0.017937,0.017937
2026-06-11,0.020222,0.000000,0.020222,0.020222
2026-06-12,0.000000,0.000000,0.000000,0.000000
2026-06-13,0.000000,0.000000,0.000000,0.000000
2026-06-14,0.000000,0.000000,0.000000,0.000000
2026-06-15,0.000000,0.000000,0.000000,0.000000
2026-06-16,0.017525,0.017525,0.017525,0.017525


## Escoragem Blend3

In [20]:
rename_dict = {
    "message_scores_BVS_CUSTOM": "SCRCRDPNMGRLPFLGBCLFCREDPGV1",#
    "message_scores_HVA4": "SERASA_HVA4",
    "pessoa_idade": "age",
    "property_type": "property_type",  # peguei de fora da consulta realizada
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "rental_value": "rental_value",  # peguei de fora da consulta realizada
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [21]:
cp_final_df.groupby("message_blend3Variables_hasPreviousContracts", dropna=False).size()

message_blend3Variables_hasPreviousContracts
NaN      15211
False    91604
True      7226
dtype: int64

In [22]:
cp_final_df.groupby("message_blend3Variables_hadOverdueInvoiceInPreviousContracts", dropna=False).size()

message_blend3Variables_hadOverdueInvoiceInPreviousContracts
NaN      15211
False    96300
True      2530
dtype: int64

In [23]:
aquiii

NameError: name 'aquiii' is not defined

In [ ]:
vars_blend4 = ['SCRCRDPNMGRLPFLGBCLFCREDPGV1', 'SERASA_HVA4', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [ ]:
df_predict_blend3 = (cp_final_df.copy()).rename(columns=rename_dict)
# df_predict = df_predict[id_vars+vars_blend4]
df_predict_blend3["REGRA_BLEND_3"] = np.where(
    df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"] >= 435,
    "BLEND3",
    "E_BVS",
)

df_predict_blend3["SCORE_BVS"] = df_predict_blend3["SCRCRDPNMGRLPFLGBCLFCREDPGV1"]
df_predict_blend3["SCORE_SERASA"] = df_predict_blend3["SERASA_HVA4"]
df_predict_blend3["RENDA"] = df_predict_blend3["income"]

df_predict_blend3.head()

## Criação de Variáveis

In [ ]:
df_predict_blend3_vars = prepare_blend3_variables(df_predict_blend3)
df_predict_blend3_escorada = predict_blend3(df_predict_blend3_vars)

## Rating

In [ ]:
bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(df_predict_blend3_escorada["predict_blend3_2_to_score"], errors="coerce")

conditions = [
    bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
    bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
    bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
    bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
    bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
    bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
    bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
    bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
]

choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

df_predict_blend3_escorada["rating_manual_blend3"] = np.select(conditions, choices, default=None)

In [ ]:
bvs = pd.to_numeric(df_predict_blend3_escorada["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(df_predict_blend3_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    bvs.between(800, 1000),                                    # 800 ≤ BVS ≤ 1000  → A
    bvs.between(700, 799),                                     # 700 ≤ BVS < 800   → B
    bvs.between(620, 699),                                     # 620 ≤ BVS < 700   → C
    bvs.between(435, 619) & score.between(476, 1000),          # 435 ≤ BVS < 620, 476 ≤ pred ≤ 1000 → B
    bvs.between(435, 619) & score.between(399, 475),           # 435 ≤ BVS < 620, 399 ≤ pred < 476  → C
    bvs.between(435, 619) & score.between(387, 398),           # 435 ≤ BVS < 620, 387 ≤ pred < 399  → D
    bvs.between(435, 619) & score.between(0, 386),             # 435 ≤ BVS < 620, 0 ≤ pred < 387    → E
    bvs.between(0, 434),                                       # 0 ≤ BVS < 435    → E
]

choices = ["A", "B", "C", "B", "C", "D", "E", "E"]

df_predict_blend3_escorada["rating_json_blend3"] = np.select(conditions, choices, default=None)

## Salvar

In [ ]:
# df_predict_blend3_escorada.to_csv(ANALYTICS_DIR/"df_predict_blend3.csv", index=False)

## Escoragem Blend4

In [ ]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HVA4": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "property_type": "property_type",  # peguei de fora da consulta realizada
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "rental_value": "rental_value",  # peguei de fora da consulta realizada
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [ ]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [ ]:
df_predict = (cp_final_df.copy()).rename(columns=rename_dict)
# df_predict = df_predict[id_vars+vars_blend4]
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] >= 334,
    "BLEND4",
    "E_BVS",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

In [ ]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [ ]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [ ]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [ ]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [ ]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada
cp_final_df

In [ ]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [ ]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

# Parte 2

## Dados do Funil

In [ ]:
query = '''
WITH tb_leads AS (
  SELECT 
    rf.contract_id,
    date(rf.dt_lead) as dt_lead,
    date(cf.requested_at) as requested_at,
    date(rf.dt_proposta_iniciada) as iniciada_at,
    date(rf.dt_proposta_enviada) as enviada_at,
    date(rf.activated_at) as activated_at,
    date(rf.cancelled_at) as cancelled_at,
    date(rf.dt_saida) as dt_saida,
    cf.tipo_contrato,
    rd.product_nm,
    cf.bureau_nm,
    CASE 
      WHEN cf.bureau_nm = 'BLEND_REGRESSAO_2026' AND CAST(ca.bvs_cust_score_nr AS FLOAT64) >= 413 THEN 'BLEND_REGRESSAO_2026'
      WHEN cf.bureau_nm = 'BVS_CUSTOM' AND CAST(ca.bvs_cust_score_nr AS FLOAT64) < 413 THEN 'BLEND_REGRESSAO_2026'
      ELSE COALESCE(cf.bureau_nm, '-1') END AS bureau_nm_ajust,
    cf.modelo_blend,
    cast(rf.total_rental_value_informed_nr as FLOAT64) as total_rental_value_informed_nr,
    cast(rf.rental_value_nr as FLOAT64) as rental_value_nr,
    -- cf.agency_id,
    -- cf.contract_city_nm,
    -- cf.person_age,
    cf.qtd_proponentes,
    cf.score_imobiliaria,
    cf.person_restriction_total_value,
    ca.bvs_cust_score_nr,
    ca.blend_regressao_predict_nr,
    cf.rating_score_ds,
    rd.pre_analysis_result,
    rd.lead_elegivel,
    rd.proposta_iniciada,
    rd.proposta_enviada,
    rd.proposta_aprovada,
    rd.proposta_ativada,
    rd.is_activeted
  FROM loft-dl-fintech.cp_gold.requests_fact AS rf
  LEFT JOIN loft-dl-fintech.cp_gold.requests_dim AS rd
    ON rf.contract_id = rd.contract_id
  LEFT JOIN loft-dl-fintech.cp_gold.credit_fact AS cf
    ON rf.contract_id = cf.contract_id
  LEFT JOIN loft-dl-fintech.cp_silver.int_credit_analyses AS ca
    ON rf.contract_id = ca.contract_id
  WHERE cf.tipo_contrato = 'PF'
),
first_defaults AS (
  SELECT
    contract_id
    , DATE(MIN(pendency_created_at)) AS first_comunicacao_date
    , DATE(MIN(pendency_at)) AS first_competencia_date
  FROM loft-dl-fintech.cp_gold.watchlist_fact
  WHERE pendency_type = "Inadimplência"
  GROUP BY contract_id
),
consulta_realizada AS(
  SELECT 
    CAST(id_externo AS INT64) AS contract_id,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.rendaConsiderada') AS FLOAT64) AS rendaConsiderada,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.rendaConsideradaContrato') AS FLOAT64) AS rendaConsideradaContrato,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.rendaUtilizada') AS rendaUtilizada,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.qtdeRestricoesContrato') AS FLOAT64) AS qtdeRestricoesContrato,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.valorRestricaoTotal') AS FLOAT64) AS valorRestricaoTotal,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.hasPreviousContracts') AS blend32_hasPreviousContracts,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.hadOverdueInvoiceInPreviousContracts') AS blend32_hadOverdueInvoiceInPreviousContracts,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.cityPc4HistoryOver100Contracts') AS FLOAT64) AS blend32_cityPc4HistoryOver100Contracts,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.blend3Variables.realEstatePc4HistoryOver100Contracts') AS FLOAT64) AS blend32_realEstatePc4HistoryOver100Contracts,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.BVS_CUSTOM') AS FLOAT64) AS bvs_custom_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.HVA3') AS FLOAT64) AS hva3_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.HVA4') AS FLOAT64) AS hva4_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.BLEND_REGRESSAO_2026') AS FLOAT64) AS blend_regressao_2026_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.scores.BLEND3_2') AS FLOAT64) AS blend3_2_score,
    cast(JSON_EXTRACT_SCALAR(json_retornado, '$.message.blendRegressaoPredict') AS FLOAT64) AS blend_regressao_predict,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.BVS_CUSTOM') AS bvs_custom_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.HVA3') AS hva3_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.HVA4') AS hva4_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.BLEND_REGRESSAO_2026') AS blend_regressao_2026_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.ratings.BLEND3_2') AS blend3_2_rating,
    JSON_EXTRACT_SCALAR(json_retornado, '$.message.errosTecnicos.0') as errosTecnicos,
    letra,
    ROW_NUMBER() OVER (PARTITION BY CAST(id_externo AS INT64) ORDER BY data DESC) as rn
  FROM `loft-dl-fintech.bronze_credpago.consulta_realizada` cr
  where REGEXP_CONTAINS(id_externo, r'^[0-9]+$')
)
SELECT
  l.*
  , case when bureau_nm_ajust = 'BLEND_REGRESSAO' then 
        case when bvs_cust_score_nr >= 800 then 'A'
            when bvs_cust_score_nr >= 700 then 'B'
            when bvs_cust_score_nr >= 625 then 'C'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.12) then 'B'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.17) then 'C'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.271) then 'D'
            when (bvs_cust_score_nr >= 478 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr > 0.271) then 'E'
            when bvs_cust_score_nr < 478 then 'E' else '-1' end
      when bureau_nm_ajust in ('BLEND_REGRESSAO_2026') then
        case when bvs_cust_score_nr >= 800 then 'A'
            when bvs_cust_score_nr >= 700 then 'B'
            when bvs_cust_score_nr >= 625 then 'C'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.132) then 'B'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.184) then 'C'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr <= 0.271) then 'D'
            when (bvs_cust_score_nr >= 413 and bvs_cust_score_nr < 625 and blend_regressao_predict_nr > 0.271) then 'E'
            when bvs_cust_score_nr < 413 then 'E' else '-1' end
      when bureau_nm_ajust = 'BLEND3_2' then 
        case when bvs_cust_score_nr >= 800 then 'A'
            when bvs_cust_score_nr >= 745 then 'B'
            when bvs_cust_score_nr >= 660 then 'C'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr >= 470) then 'B'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr >= 395) then 'C'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr >= 366) then 'D'
            when (bvs_cust_score_nr >= 470 and bvs_cust_score_nr < 660 and blend_regressao_predict_nr < 366) then 'E'
            when bvs_cust_score_nr < 470 then 'E' else '-1' end
      when bureau_nm_ajust = 'BLEND3_3' then 
        case when bvs_cust_score_nr >= 800 then 'A'
            when date(requested_at) >= date('2026-04-20') AND bvs_cust_score_nr >= 700 THEN 'B'
            when bvs_cust_score_nr >= 750 then 'B'
            when bvs_cust_score_nr >= 620 then 'C'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr >= 476) then 'B'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr >= 399) then 'C'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr >= 387) then 'D'
            when (bvs_cust_score_nr >= 435 and bvs_cust_score_nr < 620 and blend_regressao_predict_nr < 387) then 'E'
            when bvs_cust_score_nr < 435 then 'E' else '-1' end
  else 'Outros' end as rating
  , fd.first_comunicacao_date
  , fd.first_competencia_date
  , cr.rendaConsiderada
  , cr.rendaConsideradaContrato
  , cr.rendaUtilizada
  , cr.qtdeRestricoesContrato
  , cr.valorRestricaoTotal
  , cr.blend32_hasPreviousContracts
  , cr.blend32_hadOverdueInvoiceInPreviousContracts
  , cr.blend32_cityPc4HistoryOver100Contracts
  , cr.blend32_realEstatePc4HistoryOver100Contracts
  , cr.bvs_custom_score
  , cr.hva3_score
  , cr.hva4_score
  , cr.blend_regressao_2026_score
  , cr.blend3_2_score
  , cr.blend_regressao_predict
  , cr.bvs_custom_rating
  , cr.hva3_rating
  , cr.hva4_rating
  , cr.blend_regressao_2026_rating
  , cr.blend3_2_rating
  , cr.errosTecnicos
  , cr.letra
FROM tb_leads AS l
LEFT JOIN first_defaults as fd
  on l.contract_id = fd.contract_id
  and l.activated_at is not null
LEFT JOIN consulta_realizada as cr
  on l.contract_id = cr.contract_id
  and cr.rn = 1
where 1=1
    and date(l.requested_at) >= date('2026-05-01')
    and date(l.requested_at) < date(current_date())
'''

In [ ]:
df_funil = pandas_gbq.read_gbq(query, project_id=project_id)
df_funil

In [ ]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

df_funil["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

In [ ]:
bvs = pd.to_numeric(df_funil["bvs_cust_score_nr"], errors="coerce")
score = pd.to_numeric(df_funil["blend_regressao_predict_nr"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "9.E.BVS",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "2.A",
    "3.B+",
    "3.B+",
    "4.B",
    "4.B",
    "5.C",
    "6.D+",
    "7.D",
    "7.D",
    "8.E",
    "8.E",
]

df_funil["rating_pol_blend4"] = np.select(conditions, choices, default=None)

In [ ]:
df_funil.groupby("rating_pol_blend4", dropna=False).size()

quem é esse NaN?

In [ ]:
df_funil_blend4 = df_funil[df_funil["bureau_nm_ajust"] == "BLEND3_3"].copy()
df_funil_blend4["bureau_nm_ajust"] = df_funil_blend4["bureau_nm_ajust"].replace("BLEND3_3", "BLEND4")

In [ ]:
df_funil_blend4 = pd.concat([df_funil_blend4, df_funil])
df_funil_blend4.to_csv(ANALYTICS_DIR/"df_funil_blend4.csv", index=False)

In [ ]:
df_funil.to_csv(ANALYTICS_DIR/"df_funil_blend3.csv", index=False)

credit_fact: 
    pre_analysis_result

requests_fact:
    pre_analysis_result/analysis_status

requests_dim:
    pre_analysis_result/elegivel_decisao

# Modelos

Verificar última e ir apendando. Sempre D-1 (Credpago). Não dá para fazer isso na GOLD.

Focar no Blend4
Consistência, sem muito histórico.

Mudar a query
Uma única linha que tenha batido no blend4 e verificá-la, independente se depois eu bater e rodar blend4
Tudo na consulta_realizada que for blend4.

Gold possui loucuras (virada do modelo).

Monitor do Modelo: Apenas consulta realizada

Monitor do Modelo com dados do Funil: Aí entram dados da gold.

#
Para cada modelo ver: proporção de result_pre_analise